# Camada Bronze - Brazilian E-Commerce (Olist) + Meteorologia INMET (Enzo)

Carrega os ficheiros originais para o HDFS e cria as tabelas Bronze em formato Delta, na stack comum do grupo.
Os dados ficam tal como vêm da fonte (todas as colunas como texto, com os nomes originais),
com os metadados de ingestão `_data_ingestao` e `_ficheiro_origem`.

| Dataset | Ficheiro | Tabela Bronze |
|---|---|---|
| Olist (principal) | olist_orders_dataset.csv | bronze.olist_orders |
| Olist | olist_order_items_dataset.csv | bronze.olist_order_items |
| Olist | olist_order_reviews_dataset.csv | bronze.olist_order_reviews |
| Olist | olist_customers_dataset.csv | bronze.olist_customers |
| Olist | olist_sellers_dataset.csv | bronze.olist_sellers |
| Olist | olist_products_dataset.csv | bronze.olist_products |
| Olist | product_category_name_translation.csv | bronze.olist_product_category_name_translation |
| Brazil temperature 2012-2019 / INMET (complementar) | output_data_part1.csv + output_data_part2.csv | bronze.inmet_observacoes |

Fontes: Kaggle `olistbr/brazilian-ecommerce` e `fabiojorge/brazil-temperature-2012-2019`.
Não são carregados os ficheiros da Olist que o modelo não usa (pagamentos e geolocalização).

**Leitura dos CSV:** as avaliações têm comentários com mudanças de linha e aspas, por isso todos os CSV são lidos com
`multiLine` e `escape='"'` e sem `inferSchema` (assim os códigos postais não perdem os zeros à esquerda).
Os ficheiros do INMET usam `;` como separador.

In [1]:
ficheiros = {
    "/home/jovyan/work/data/enzo/olist_orders_dataset.csv": "/projeto/bronze/enzo/olist_orders_dataset.csv",
    "/home/jovyan/work/data/enzo/olist_order_items_dataset.csv": "/projeto/bronze/enzo/olist_order_items_dataset.csv",
    "/home/jovyan/work/data/enzo/olist_order_reviews_dataset.csv": "/projeto/bronze/enzo/olist_order_reviews_dataset.csv",
    "/home/jovyan/work/data/enzo/olist_customers_dataset.csv": "/projeto/bronze/enzo/olist_customers_dataset.csv",
    "/home/jovyan/work/data/enzo/olist_sellers_dataset.csv": "/projeto/bronze/enzo/olist_sellers_dataset.csv",
    "/home/jovyan/work/data/enzo/olist_products_dataset.csv": "/projeto/bronze/enzo/olist_products_dataset.csv",
    "/home/jovyan/work/data/enzo/product_category_name_translation.csv": "/projeto/bronze/enzo/product_category_name_translation.csv",
    "/home/jovyan/work/data/enzo/output_data_part1.csv": "/projeto/bronze/enzo/inmet/output_data_part1.csv",
    "/home/jovyan/work/data/enzo/output_data_part2.csv": "/projeto/bronze/enzo/inmet/output_data_part2.csv",
}

In [2]:
# Copiar os ficheiros originais para a zona Bronze do HDFS (full load: apaga a versão anterior e volta a carregar)
from hdfs import InsecureClient
client = InsecureClient("http://hdfs-nn:9870", user="anonymous")

for from_path, to_path in ficheiros.items():
    client.delete(to_path)
    client.upload(to_path, from_path)
    print(to_path, client.status(to_path)["length"], "bytes")

/projeto/bronze/enzo/olist_orders_dataset.csv 17654914 bytes
/projeto/bronze/enzo/olist_order_items_dataset.csv 15438671 bytes
/projeto/bronze/enzo/olist_order_reviews_dataset.csv 14451670 bytes
/projeto/bronze/enzo/olist_customers_dataset.csv 9033957 bytes
/projeto/bronze/enzo/olist_sellers_dataset.csv 174703 bytes
/projeto/bronze/enzo/olist_products_dataset.csv 2379446 bytes
/projeto/bronze/enzo/product_category_name_translation.csv 2613 bytes
/projeto/bronze/enzo/inmet/output_data_part1.csv 200674760 bytes
/projeto/bronze/enzo/inmet/output_data_part2.csv 92595568 bytes


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window
from delta import *

# Stack comum do grupo: HDFS (hdfs-nn), Hive Metastore e Delta Lake 2.4
warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

builder = SparkSession \
    .builder \
    .appName("Bronze - Olist + INMET (Enzo)") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.session.timeZone", "UTC") \
    .enableHiveSupport()

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Os nomes originais das colunas têm espaços e caracteres especiais;
# o column mapping do Delta permite guardá-los tal como vêm da fonte
spark.conf.set("spark.databricks.delta.properties.defaults.columnMapping.mode", "name")
spark.conf.set("spark.databricks.delta.properties.defaults.minReaderVersion", "2")
spark.conf.set("spark.databricks.delta.properties.defaults.minWriterVersion", "5")

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze LOCATION 'hdfs://hdfs-nn:9000/warehouse/bronze.db/'")

def gravar_bronze(df, tabela):
    """Acrescenta os metadados de ingestão e grava a tabela Delta tal como veio da fonte."""
    df = df.withColumn("_data_ingestao", F.current_timestamp()) \
           .withColumn("_ficheiro_origem", F.input_file_name())
    df.write \
        .mode("overwrite") \
        .format("delta") \
        .option("overwriteSchema", "true") \
        .option("path", f"hdfs://hdfs-nn:9000/warehouse/bronze.db/{tabela}/") \
        .saveAsTable(f"bronze.{tabela}")
    print(f"bronze.{tabela}: {spark.table(f'bronze.{tabela}').count()} linhas, {len(df.columns) - 2} colunas originais")

def ler_csv(hdfs_path, **opcoes):
    """Lê um CSV do HDFS com todas as colunas como texto (sem inferSchema)."""
    leitor = spark.read.option("header", "true").option("multiLine", "true").option("escape", '"')
    for k, v in opcoes.items():
        leitor = leitor.option(k, v)
    return leitor.csv("hdfs://hdfs-nn:9000" + hdfs_path)

In [5]:
olist = ['olist_orders_dataset', 'olist_order_items_dataset', 'olist_order_reviews_dataset', 'olist_customers_dataset', 'olist_sellers_dataset', 'olist_products_dataset', 'product_category_name_translation']
for f in olist:
    tabela = f if f.startswith("olist_") else "olist_" + f
    gravar_bronze(ler_csv(f"/projeto/bronze/enzo/{f}.csv"), tabela.replace("_dataset", ""))

# INMET: os dois ficheiros vão para a mesma tabela (o _ficheiro_origem distingue-os)
gravar_bronze(ler_csv("/projeto/bronze/enzo/inmet/*.csv", sep=";", multiLine="false"), "inmet_observacoes")

bronze.olist_orders: 99441 linhas, 8 colunas originais
bronze.olist_order_items: 112650 linhas, 7 colunas originais
bronze.olist_order_reviews: 99224 linhas, 7 colunas originais
bronze.olist_customers: 99441 linhas, 5 colunas originais
bronze.olist_sellers: 3095 linhas, 4 colunas originais
bronze.olist_products: 32951 linhas, 9 colunas originais
bronze.olist_product_category_name_translation: 71 linhas, 2 colunas originais
bronze.inmet_observacoes: 1845665 linhas, 28 colunas originais


## Verificação

As tabelas Bronze têm de ter o mesmo número de registos dos ficheiros originais (contados com o leitor de CSV do Python, que respeita as mudanças de linha dentro de aspas).

In [6]:
import csv
def registos(caminho, sep=","):
    with open(caminho, encoding="utf-8", newline="") as f:
        return sum(1 for _ in csv.reader(f, delimiter=sep)) - 1

for f in olist:
    tabela = (f if f.startswith("olist_") else "olist_" + f).replace("_dataset", "")
    n, real = registos(f"/home/jovyan/work/data/enzo/{f}.csv"), spark.table(f"bronze.{tabela}").count()
    print(f"{tabela:42s} ficheiro {n:>8}  bronze {real:>8}  {'OK' if n == real else 'DIFERENTE'}")
    assert n == real

n = sum(registos(f"/home/jovyan/work/data/enzo/output_data_part{i}.csv", sep=";") for i in (1, 2))
real = spark.table("bronze.inmet_observacoes").count()
print(f"{'inmet_observacoes':42s} ficheiro {n:>8}  bronze {real:>8}  {'OK' if n == real else 'DIFERENTE'}")
assert n == real

olist_orders                               ficheiro    99441  bronze    99441  OK
olist_order_items                          ficheiro   112650  bronze   112650  OK
olist_order_reviews                        ficheiro    99224  bronze    99224  OK
olist_customers                            ficheiro    99441  bronze    99441  OK
olist_sellers                              ficheiro     3095  bronze     3095  OK
olist_products                             ficheiro    32951  bronze    32951  OK
olist_product_category_name_translation    ficheiro       71  bronze       71  OK
inmet_observacoes                          ficheiro  1845665  bronze  1845665  OK


In [7]:
spark.table("bronze.olist_order_reviews").select("review_id", "order_id", "review_score", "review_creation_date").show(3, truncate=False)
spark.table("bronze.inmet_observacoes").select("CodigoOMM", "Estado", "Data", "Hora", "Precipitacao", "TempBulboSeco", "UmidadeRelativa").show(3, truncate=False)

+--------------------------------+--------------------------------+------------+--------------------+
|review_id                       |order_id                        |review_score|review_creation_date|
+--------------------------------+--------------------------------+------------+--------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |2018-01-18 00:00:00 |
|80e641a11e56f04c1ad469d5645fdfde|a548910a1c6147796b98fdf73dbeba33|5           |2018-03-10 00:00:00 |
|228ce5500dc1d8e020d8d1322874b6f0|f9e4b658b201a9f2ecdecbb34bed034b|5           |2018-02-17 00:00:00 |
+--------------------------------+--------------------------------+------------+--------------------+
only showing top 3 rows

+---------+------+----------+----+------------+-------------+---------------+
|CodigoOMM|Estado|Data      |Hora|Precipitacao|TempBulboSeco|UmidadeRelativa|
+---------+------+----------+----+------------+-------------+---------------+
|83097    |SE    |09/11/201

In [8]:
spark.stop()